# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

To prioritse pages, we compute a Final Refresh Score (on a 0 to 100 scale) by blending:

70% of the Model's Predicted Probability of Decline (identifying risk)
30% of the Normalised Heuristic Baseline Score (identifying visibility and content gaps)
By blending these, to ensure that we prioritise pages that are highly likely to decline AND get high traffic (so fixing them makes a huge difference).


Based on the reason codes, here are specific action for each page:

expand_and_refresh: Triggered by thin_visible_page. The article is too short (under 1200 words); editors should add subtopics, details, and examples.
refresh_and_review_ctr: Triggered by low_ctr_visible_page combined with decline. The page ranks well but click-through rates are low. Editors should refresh the text and optimise the meta title/description.
refresh_and_review_engagement: Triggered by low_engagement_visible_page combined with decline. Users are landing on the page but leaving quickly. Editors should improve layout, add media, and rewrite the introduction.
refresh: General update of facts, links, and keywords.
monitor: Keep tracking; page is currently stable.

In [2]:
import numpy as np, pandas as pd 
data_features = pd.read_csv("../../data/processed/refresh_feature_vector.csv")
data_baseline = pd.read_csv("../../data/processed/baseline_refresh_queue.csv")
data_predictions = pd.read_csv("../../data/processed/model_predictions.csv")

# lets take a glimps of the data
display(data_features.head())
display(data_baseline.head())
display(data_predictions.head())

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,...,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,clicks_90d,sessions_90d,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction
0,content_9532f197bbc8,client_4e07408562,1,0.941189,0.999633,0.8432,0.979233,0.871347,declining_with_demand|page_one_decay_risk|low_...,refresh,...,2689,1098,2.0,0.87,8.01,28.75,445,104,0.0,down
1,content_4d1fe5b32dc2,client_19581e27de,2,0.934889,0.994167,0.8432,0.963733,0.866582,page_one_decay_risk|low_engagement_visible_page,monitor,...,512,549,2.5,0.52,7.47,13.15,329,104,0.0,stable
2,content_07f2e7a6f38a,client_19581e27de,3,0.934080,0.994467,0.8432,0.959965,0.866843,page_one_decay_risk|low_engagement_visible_page,monitor,...,856,780,2.7,0.85,2.05,4.60,313,104,0.0,stable
3,content_e5ae436f9a16,client_4e07408562,4,0.933606,0.996000,0.8432,0.955347,0.868180,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,533,522,3.0,0.45,7.09,12.60,421,104,0.0,stable
4,content_3430a8b94511,client_19581e27de,5,0.933559,0.998167,0.8432,0.951314,0.870069,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,...,440,534,3.3,0.29,6.18,11.04,329,104,0.0,stable


,content_id,client_id,is_declining_label,split,prob_logistic_regression,prob_decision_tree,prob_random_forest,best_model_name,best_model_probability
0,content_304f48230142,client_f369cb89fc,1,train,0.490173,0.413181,0.542529,random_forest,0.542529
1,content_a1fb4e703a9e,client_4e07408562,1,train,0.632719,0.448492,0.432100,random_forest,0.432100
2,content_9aa793d4d895,client_7f2253d7e2,1,train,0.705139,0.694024,0.814868,random_forest,0.814868
3,content_331d6c4de07b,client_19581e27de,0,train,0.274520,0.316977,0.348316,random_forest,0.348316
4,content_d99b7a2d90ca,client_3fdba35f04,1,train,0.379525,0.694024,0.683803,random_forest,0.683803


In [3]:
# merge the features with predictions 
final_data = data_baseline.merge(data_predictions[["content_id", "best_model_name", "best_model_probability"]], 
                    on="content_id", how="left")

# normalise baseline score
baseline_min = final_data['baseline_refresh_score'].min()
baseline_max = final_data['baseline_refresh_score'].max()
final_data['baseline_score_normalise'] = (final_data['baseline_refresh_score'] - baseline_min) / (baseline_max - baseline_min)

final_data['final_refresh_score'] = (100 * (0.70 * final_data['best_model_probability'] + 0.30 * final_data['baseline_score_normalise'])).clip(0, 100)

final_data = final_data.sort_values("final_refresh_score", ascending=False).reset_index(drop=True)
final_data['final_rank'] = final_data.index + 1

print("Top 10 prioeitisong features")
display(final_data.head())
display(final_data[['final_rank', 'content_id', 'final_refresh_score', "best_model_probability", "is_declining_label"]].head(10))


Top 10 prioeitisong features


,content_id,client_id,baseline_rank,baseline_refresh_score,visibility_score,freshness_risk_score,position_opportunity_score,depth_gap_score,reason_codes,suggested_action_baseline,...,scroll_rate,content_age_days,days_since_last_update,word_count,trend_direction,best_model_name,best_model_probability,baseline_score_normalise,final_refresh_score,final_rank
0,content_1f080331fa2b,client_3fdba35f04,786,0.844481,0.905067,0.8432,0.797936,0.600210,declining_with_demand|low_ctr_visible_page|low...,refresh_and_review_ctr,...,10.59,165,104,1404.0,down,random_forest,0.783472,0.896373,81.734212,1
1,content_d6570c51c9bd,client_3fdba35f04,4516,0.695884,0.695000,0.8432,0.565929,0.468835,declining_with_demand|low_ctr_visible_page,refresh_and_review_ctr,...,13.33,165,104,1362.0,down,random_forest,0.849842,0.737144,81.603243,2
2,content_6aa43079fb0c,client_3fdba35f04,1044,0.825477,0.856900,0.8432,0.807934,0.555471,declining_with_demand|low_ctr_visible_page,refresh_and_review_ctr,...,2.86,139,104,1457.0,down,random_forest,0.789490,0.876010,81.544618,3
3,content_72e800a9c214,client_3fdba35f04,817,0.842545,0.911433,0.8432,0.777508,0.612696,declining_with_demand|low_ctr_visible_page,refresh_and_review_ctr,...,4.88,139,104,1371.0,down,random_forest,0.776297,0.894299,81.169731,4
4,content_e04eb9549989,client_3fdba35f04,2538,0.749468,0.741233,0.8432,0.701903,0.490771,declining_with_demand|low_ctr_visible_page,refresh_and_review_ctr,...,28.57,131,104,1408.0,down,random_forest,0.816010,0.794562,80.957565,5


,final_rank,content_id,final_refresh_score,best_model_probability,is_declining_label
0,1,content_1f080331fa2b,81.734212,0.783472,1
1,2,content_d6570c51c9bd,81.603243,0.849842,1
2,3,content_6aa43079fb0c,81.544618,0.789490,1
3,4,content_72e800a9c214,81.169731,0.776297,1
4,5,content_e04eb9549989,80.957565,0.816010,1
5,6,content_b69288c5e701,80.798090,0.796332,1
6,7,content_9b6df29f7889,80.650656,0.846499,1
7,8,content_ba6f9dfcbca1,80.432641,0.827496,1
8,9,content_4d76cdb3387b,80.428403,0.844030,1
9,10,content_b4f35d640b1c,80.428243,0.845325,1


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended Use:** This playbook is a decision-support tool for content marketing and editorial teams. It helps managers allocate their weekly writing resources to high-impact pages that are losing search visibility.
* **Limits:**
No Algorithm Guarantee: The model cannot predict or adapt to sudden Google Core updates.
No Seasonality: The 90-day static window cannot predict traffic changes caused by calendar seasonality (e.g., holiday topics).
Cold Start Problem: The model is not valid for brand-new websites with zero historical search or analytics data.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

* **Human Review Checklist:** Before making changes, an editor must manually verify:
Trend validation: Did the traffic decline because of search intent shifts, or did a competitor simply run a paid ad campaign that temporarily pushed us down?
Relevance: Is the page still aligned with the brand's current business goals?
Technical Check: Is the drop in CTR caused by broken page layouts or slow page speeds rather than the text content?
* **The No-Go List:**
Automatic Publishing: Never allow an LLM to auto-write and auto-publish content updates without human proofreading.
Automated Deletions: Never allow a script to automatically delete or redirect underperforming pages, as this can break internal link structures and destroy remaining domain authority.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

* **Monitoring Metrics:**
Track Precision@50 on live recommendations: out of the top 50 pages updated, what percentage successfully stopped declining or recovered?
Track client-level decay rate (the proportion of total portfolio pages in decline).
* **Retrain Triggers:**
Performance Decay: If Precision@50 drops by more than 15% compared to our validation benchmarks, the model's rules have gone stale.
Concept Drift: Following any major Google Search Core Update, we must collect fresh 90-day metrics and retrain the model to capture the new search engine behavior.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
# Map the reason codes and suggest actions

def get_final_reasons(row):
    reasons = [r for r in str(row.get("reason_codes", "")).split("|") if r and r != "nan"]
    if row['best_model_probability'] >= 0.65:
        reasons.append("model_decline_risk")
    if row['best_model_probability'] >= 0.5 and row['best_model_probability'] >= 500:
        reasons.append("visible_model_opportunity")
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append("ctr_review_candidate")
    return "|".join(set(reasons))


final_data['final_reason_codes'] = final_data.apply(get_final_reasons, axis=1)

def get_final_action(row):
    reasons = set(row['final_reason_codes'].split("|"))

    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "ctr_review_candidate" in reasons and ("model_decline_risk" in reasons or "declining_with_demand" in reasons):
        return "refresh and review_ctr"
    if {"model_decline_risk", "declining_with_demand", "stale_visible_page", "visible_model_opportunity"}.intersection(reasons):
        return "refresh"
    return "monitor"


final_data['suggested_action'] = final_data.apply(get_final_action, axis=1)



In [5]:
# assign confidence categories
high_threshold = final_data['final_refresh_score'].quantile(0.8)
medium_threshold = final_data['final_refresh_score'].quantile(0.5)

def get_confidence(row):
    if (row['final_refresh_score'] >= high_threshold and row['impressions_90d'] >= 500 and row['best_model_probability'] >= 0.5):
        return "high"
    if row['final_refresh_score'] >= medium_threshold:
        return "medium"
    return "low"


final_data['confidence'] = final_data.apply(get_confidence, axis=1)

# save the final queue
output_columns = [
    "final_rank", "content_id", "client_id", "final_refresh_score", 
    "confidence", "suggested_action", "final_reason_codes", 
    "is_declining_label", "impressions_90d", "avg_position", "ctr"
]

data_export = final_data[output_columns]

data_export.to_csv("../outputs/refresh_queue.csv", index=False)
print("Saved Final queue to ../output/refresh_queue.csv")

Saved Final queue to ../output/refresh_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.